In [0]:
# Inicialización de parámetros
dbutils.widgets.text("workspace", "workspace", "workspace")
dbutils.widgets.text("schema", "hiraoka", "schema")
dbutils.widgets.text("type_of_load", "Histórica", "type of load")

# Extraer valores de los parámetros
workspace = dbutils.widgets.get("workspace")
schema = dbutils.widgets.get("schema")
type_of_load = dbutils.widgets.get("type_of_load")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
# Leer tabla bronze
table_silver = f"{workspace}.{schema}.silver_celulares"

df_silver = spark.read.table(table_silver).withColumn("FechaActualizacion", current_timestamp())

In [0]:
from delta.tables import DeltaTable

In [0]:
table_gold = f"{workspace}.{schema}.gold_celulares"

In [0]:
if type_of_load == "Histórica":
    df_silver.write.mode("overwrite").saveAsTable(table_gold)
else:
    delta_gold = DeltaTable.forName(spark, table_gold)
    (
        delta_gold.alias("T").merge(df_silver.alias("S"), "T.Codigo = S.Codigo")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )